In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import pearsonr
import math
import pydicom
from show_keypoints import show_keypoints
import matplotlib.cm as cm
import scipy.stats as stats
from PIL import Image, ImageDraw
import os
import re
from sklearn.metrics import r2_score

## get all phenotypes

In [ ]:
def cal_euclidean_distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

### Length phenotyps

#### From hip

In [ ]:
# all prediciton results
all_pred = json.load(open('key_results/all_res_pred_on_pred_23.json'))

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

# define the columns
columns = []
for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        columns.append(hip_regions[i] + '2' + hip_regions[j])

# define the index
index = sorted(all_pred.keys())

# create a dataframe
hip_dist_df = pd.DataFrame(index = index, columns = columns)

# calculate the distance
for idx in list(all_pred.keys()):
    for m in range(len(hip_regions)):
        for n in range(m+1, len(hip_regions)):
            ed = cal_euclidean_distance(all_pred[idx][m][0],
                                        all_pred[idx][m][1],
                                        all_pred[idx][n][0],
                                        all_pred[idx][n][1])
            hip_dist_df.loc[idx, hip_regions[m] + '2' + hip_regions[n]] = ed

In [ ]:
hip_dist_df

#### From head

In [ ]:
head_pred = json.load(open('key_results/all_res_pred_on_pred_head.json'))

In [ ]:
hip_dist_df['ear_left2ear_right'] = np.nan

for idx in list(head_pred.keys()):
    ed = cal_euclidean_distance(head_pred[idx][3][0],
                                head_pred[idx][3][1],
                                head_pred[idx][4][0],
                                head_pred[idx][4][1])
    hip_dist_df.loc[idx, 'ear_left2ear_right'] = ed

In [ ]:
hip_dist_df

#### Hip height

In [ ]:
def calculate_height(x1, y1, x2, y2, x3, y3, x4, y4):
    upper_center = ((x1 + x2) / 2, (y1 + y2) / 2)
    lower_center = ((x3 + x4) / 2, (y3 + y4) / 2)
    return math.sqrt((upper_center[0] - lower_center[0]) ** 2 + (upper_center[1] - lower_center[1]) ** 2)

In [ ]:
# all prediciton results
all_pred = json.load(open('key_results/all_res_pred_on_pred_23.json'))

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

ul = hip_regions.index('iliac_crest_left')
ur = hip_regions.index('iliac_crest_right')
ll = hip_regions.index('inferior_pubic_ramus_left')
lr = hip_regions.index('inferior_pubic_ramus_right')

hip_dist_df['hip_height'] = np.nan
for idx in hip_dist_df.index:
    kps = all_pred[idx]
    hip_height = calculate_height(kps[ul][0],  kps[ul][1], 
                                  kps[ur][0],  kps[ur][1], 
                                  kps[ll][0],  kps[ll][1], 
                                  kps[lr][0],  kps[lr][1])
    hip_dist_df.loc[idx, 'hip_height'] = hip_height

In [ ]:
hip_dist_df

#### bi-acetabular width

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

iel = hip_regions.index('iliopubic_eminence_left')
ier = hip_regions.index('iliopubic_eminence_right')
ail = hip_regions.index('acetabular_inferior_left')
air = hip_regions.index('acetabular_inferior_right')

hip_dist_df['bi_acetabular_width'] = np.nan
for idx in hip_dist_df.index:
    kps = all_pred[idx]
    bi_acetabular_width = calculate_height(kps[iel][0],  kps[iel][1], 
                                           kps[ail][0],  kps[ail][1], 
                                           kps[ier][0],  kps[ier][1], 
                                           kps[air][0],  kps[air][1])
    hip_dist_df.loc[idx, 'bi_acetabular_width'] = bi_acetabular_width

In [ ]:
hip_dist_df

In [ ]:
hip_dist_df.to_csv('key_results/hip_dist_pixel_23.csv')

### convert pixels to cm

#### Get beta to convert pixels to cm - get beta

In [ ]:
# all prediciton results
# all_pred = json.load(open('key_results/all_res_pred_on_pred_23.json'))

# load regression results
beta_df = pd.read_excel("../UKB_xray_image_info/HPE_height_pixel_length_betas.xlsx")[['Pixel Lengths', 'Value']]

# load image size results
img_size_df = pd.read_csv("../UKB_xray_image_info/dcm_size.csv")

In [ ]:
img_size_df

In [ ]:
# process 960x384 and 816x288 separately

# 960x384
img_size_df_93 = img_size_df[img_size_df['image_id'].str.contains('_93')]
img_beta_df_93 = img_size_df_93.merge(beta_df, left_on='origin_image_size', right_on='Pixel Lengths', how='inner').drop(columns=['Pixel Lengths'])

# 816x288
beta_82 = beta_df[beta_df['Pixel Lengths'] == "864x288"]['Value'].values[0]
img_size_df_82 = img_size_df[img_size_df['image_id'].str.contains('_82')]
img_size_df_82['Value'] = beta_82
img_beta_df_82 = img_size_df_82

# combine
img_beta_df = pd.concat([img_beta_df_93, img_beta_df_82], axis=0)

In [ ]:
img_beta_df.to_csv('../UKB_xray_image_info/img_beta_df.csv', index=False)

#### Calculate cm

In [ ]:
img_beta_df = pd.read_csv('../UKB_xray_image_info/img_beta_df.csv')

hip_dist_df = pd.read_csv('key_results/hip_dist_pixel_23.csv', index_col=0)

In [ ]:
# calculate cm length
hip_dist_df_cm = pd.DataFrame(index=img_beta_df['image_id'].tolist(), columns=hip_dist_df.columns)

for idx in hip_dist_df_cm.index:
    hip_dist_df_cm.loc[idx, :] = hip_dist_df.loc[idx, :] * img_beta_df[img_beta_df['image_id'] == idx]['Value'].values[0]

In [ ]:
hip_dist_df_cm

In [ ]:
hip_dist_df_cm.to_csv('key_results/hip_dist_df_cm_23.csv')

#### Add body phenotype

In [ ]:
hip_dist_df_cm = pd.read_csv('key_results/hip_dist_df_cm_23.csv', index_col=0)
body_dist_df_cm = pd.read_csv('../left_vs_right/pred_res/pheno_cm_length.csv', index_col=0)

hip_dist_df_cm = hip_dist_df_cm.merge(body_dist_df_cm, left_index=True, right_index=True, how='inner')

In [ ]:
hip_dist_df_cm

In [ ]:
hip_dist_df_cm.to_csv('key_results/hip_body_dist_df_cm.csv')

#### add patient info

In [ ]:
hip_dist_df_cm = pd.read_csv('key_results/hip_body_dist_df_cm.csv', index_col=0)
all_dcm_info = pd.read_csv("all_prediction/all_dcm_info.csv")[['image_id', 'file_name', 'Patient EID', 'p_sex', 'p_age', 'p_weight', 'p_height']]

In [ ]:
pwd

In [ ]:
hip_pheno = all_dcm_info.merge(hip_dist_df_cm, left_on='image_id', right_index=True, how='inner'); hip_pheno

In [ ]:
hip_pheno.set_index('image_id', inplace=True)

#### phenotypes of ratio

In [ ]:
hip_pheno['sacrum_ratio_w2h'] = hip_pheno['sciatic_notch_left2sciatic_notch_right'] / hip_pheno['sacrum2pubic_tubercle']
hip_pheno['hip_ratio_w2h'] = hip_pheno['iliac_spine_left2iliac_spine_right'] / hip_pheno['hip_height']
hip_pheno['sacrum2pubic_tubercle_devide_ear_left2ear_right'] = hip_pheno['sacrum2pubic_tubercle'] / hip_pheno['ear_left2ear_right']
hip_pheno['sacrum_left2sacrum_right_devide_ear_left2ear_right'] = hip_pheno['sacrum_left2sacrum_right'] / hip_pheno['ear_left2ear_right']
hip_pheno['sciatic_notch_left2sciatic_notch_right_divide_ear_left2ear_right'] = hip_pheno['sciatic_notch_left2sciatic_notch_right'] / hip_pheno['ear_left2ear_right']

# Added on May 17th, 2023 
# 1. iliac flare
# 1.1 sacrum_left to sacrum_right / hip width
hip_pheno['sacrum_left2sacrum_right_divide_iliac_spine_left2iliac_spine_right'] = \
    hip_pheno['sacrum_left2sacrum_right'] / hip_pheno['iliac_spine_left2iliac_spine_right']

# 1.2 sciatic_notch_left to sciatic_notch_right / hip width
hip_pheno['sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right'] = \
    hip_pheno['sciatic_notch_left2sciatic_notch_right'] / hip_pheno['iliac_spine_left2iliac_spine_right']

# 1.3 inferior_iliac_spine_left to inferior_iliac_spine_right / hip width
hip_pheno['inferior_iliac_spine_left2inferior_iliac_spine_right_divide_iliac_spine_left2iliac_spine_right'] = \
    hip_pheno['inferior_iliac_spine_left2inferior_iliac_spine_right'] / hip_pheno['iliac_spine_left2iliac_spine_right']

# Added on July 13th, 2023
# arm to torso ratio
hip_pheno['arm_devide_torso'] = ((hip_pheno['arm_123_left'] + hip_pheno['arm_123_right']) / 2) / hip_pheno['torso_length']

In [ ]:
hip_pheno

#### add angle to phenotype df

In [ ]:
def calculate_angle(x1, y1, x2, y2, x3, y3):
    """Calculate the angle between three points."""
    # x1, y1 should be the angle point
    b = math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
    c = math.sqrt((x1 - x3) ** 2 + (y1 - y3) ** 2)
    a = math.sqrt((x2 - x3) ** 2 + (y2 - y3) ** 2)
    angle = math.acos((b ** 2 + c ** 2 - a ** 2) / (2 * b * c))
    return math.degrees(angle)

calculate_angle(0, 1, 0, 0, 1, 0)

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

for idx in hip_pheno.index:
    kps = all_pred[idx]

    # calculate pubic arch angle
    iprl = hip_regions.index('inferior_pubic_ramus_left')
    iprr = hip_regions.index('inferior_pubic_ramus_right')
    pa = hip_regions.index('pubic_arch')
    pubic_arch_angle = calculate_angle(kps[pa][0], kps[pa][1], 
                                       kps[iprl][0], kps[iprl][1], 
                                       kps[iprr][0], kps[iprr][1])
    hip_pheno.loc[idx, 'pubic_arch_angle'] = pubic_arch_angle

    # calculate acetabular inclination
    iel = hip_regions.index('iliopubic_eminence_left')
    ier = hip_regions.index('iliopubic_eminence_right')
    ail = hip_regions.index('acetabular_inferior_left')
    air = hip_regions.index('acetabular_inferior_right')

    # left
    acetabular_inclination_left = calculate_angle(kps[ail][0], kps[ail][1],
                                                  kps[iel][0], kps[iel][1],
                                                  kps[air][0], kps[air][1])
    acetabular_inclination_left = 180 - acetabular_inclination_left
    hip_pheno.loc[idx, 'acetabular_inclination_left'] = acetabular_inclination_left

    # right
    acetabular_inclination_right = calculate_angle(kps[air][0], kps[air][1],
                                                   kps[ail][0], kps[ail][1],
                                                   kps[ier][0], kps[ier][1])
    
    acetabular_inclination_right = 180 - acetabular_inclination_right
    hip_pheno.loc[idx, 'acetabular_inclination_right'] = acetabular_inclination_right

    # Added on May 17th, 2023 
    # 1. angle of iliac_spine_left to pubic_tubercle to iliac_spine_right 
    isl = hip_regions.index('iliac_spine_left')
    isr = hip_regions.index('iliac_spine_right')
    pt = hip_regions.index('pubic_tubercle')
    angle_pubic_tubercle_iliac_spine_lr = calculate_angle(kps[pt][0], kps[pt][1],
                                                          kps[isl][0], kps[isl][1],
                                                          kps[isr][0], kps[isr][1])
    hip_pheno.loc[idx, 'angle_pubic_tubercle_iliac_spine_lr'] = angle_pubic_tubercle_iliac_spine_lr

    # 2. angle of iliac_spine_left to pubic_arch to iliac_spine_right
    isl = hip_regions.index('iliac_spine_left')
    isr = hip_regions.index('iliac_spine_right')
    pa = hip_regions.index('pubic_arch') 
    angle_pubic_arch_iliac_spine_lr = calculate_angle(kps[pa][0], kps[pa][1],
                                                      kps[isl][0], kps[isl][1],               
                                                      kps[isr][0], kps[isr][1])
    hip_pheno.loc[idx, 'angle_pubic_arch_iliac_spine_lr'] = angle_pubic_arch_iliac_spine_lr

In [ ]:
hip_pheno["acetabular_inclination_diff"] = hip_pheno["acetabular_inclination_left"] - hip_pheno["acetabular_inclination_right"]

In [ ]:
hip_pheno

In [ ]:
hip_pheno['p_height'] = hip_pheno['p_height'] * 100

In [ ]:
hip_pheno.to_csv('key_results/hip_pheno_23.csv')

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23.csv')

In [ ]:
fid_height = pd.read_csv("../UKB_xray_image_info/fids/fid_disease/fid_info_400k_20240104.csv")[['eid', 'standing_height_imaging_visit', 'sex', 'age_imaging_visit', 'bmi_imaging_visit', 'weight_imaging_visit']]

In [ ]:
hip_pheno = fid_height.merge(hip_pheno, left_on = 'eid', right_on='Patient EID', how='right').drop(columns=['Patient EID', 'p_height', 'p_weight', 'p_sex', 'p_age'])

In [ ]:
hip_pheno

In [ ]:
hip_pheno.isna().sum()[:10]

In [ ]:
hip_pheno = hip_pheno.dropna()

In [ ]:
hip_pheno

In [ ]:
hip_pheno.to_csv('key_results/hip_pheno_23_cm.csv', index=False)

# Get important phenotypes (According to Dr. Marianne Brasil's suggestions)

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23_cm.csv'); hip_pheno

In [ ]:
select_pheno = hip_pheno[['image_id',
                          'file_name',
                          'eid',
                          'sex',
                          'standing_height_imaging_visit',
                          'weight_imaging_visit',
                          'bmi_imaging_visit',
                          'age_imaging_visit',
                          'hip_height',
                          'iliac_spine_left2iliac_spine_right',
                          'sciatic_notch_left2sciatic_notch_right',
                          'sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right',
                          'sacrum2pubic_tubercle',
                          'sciatic_notch_left2inferior_iliac_spine_left',
                          'sciatic_notch_right2inferior_iliac_spine_right',
                          'iliopubic_eminence_left2acetabular_inferior_left',
                          'iliopubic_eminence_right2acetabular_inferior_right',
                          'angle_pubic_tubercle_iliac_spine_lr',
                          'pubic_arch_angle',
                          'acetabular_inclination_left',
                          'acetabular_inclination_right',
                          'ear_left2ear_right',
                          'trochanter_left2trochanter_right',
                          'shoulder_width',
                          'arm_devide_torso',
                          'bi_acetabular_width',
                          'arm_12_left',
                          'arm_12_right',
                          'arm_23_left',
                          'arm_23_right',
                          'arm_123_left',
                          'arm_123_right',
                          'leg_12_left',
                          'leg_12_right',
                          'leg_23_left',
                          'leg_23_right',
                          'leg_123_left',
                          'leg_123_right',
                          'torso_length']]

# calculate left-right average for some phenotypes
select_pheno['sciatic_notch2inferior_iliac_spine'] = \
    (select_pheno['sciatic_notch_left2inferior_iliac_spine_left'] + select_pheno['sciatic_notch_right2inferior_iliac_spine_right']) / 2
select_pheno['iliopubic_eminence2acetabular_inferior'] = \
    (select_pheno['iliopubic_eminence_left2acetabular_inferior_left'] + select_pheno['iliopubic_eminence_right2acetabular_inferior_right']) / 2
select_pheno['acetabular_inclination'] = \
    (select_pheno['acetabular_inclination_left'] + select_pheno['acetabular_inclination_right']) / 2
select_pheno['arm_12_length'] = \
    (select_pheno['arm_12_left'] + select_pheno['arm_12_right']) / 2
select_pheno['leg_12_length'] = \
    (select_pheno['leg_12_left'] + select_pheno['leg_12_right']) / 2
select_pheno['arm_23_length'] = \
    (select_pheno['arm_23_left'] + select_pheno['arm_23_right']) / 2
select_pheno['leg_23_length'] = \
    (select_pheno['leg_23_left'] + select_pheno['leg_23_right']) / 2
select_pheno['arm_length'] = \
    (select_pheno['arm_123_left'] + select_pheno['arm_123_right']) / 2
select_pheno['leg_length'] = \
    (select_pheno['leg_123_left'] + select_pheno['leg_123_right']) / 2
select_pheno['leg_divide_torso'] = \
    (select_pheno['leg_length'] / select_pheno['torso_length'])

select_pheno.drop(columns = ['sciatic_notch_left2inferior_iliac_spine_left', 
                             'sciatic_notch_right2inferior_iliac_spine_right', 
                             'iliopubic_eminence_left2acetabular_inferior_left', 
                             'iliopubic_eminence_right2acetabular_inferior_right',
                             'acetabular_inclination_left',
                             'acetabular_inclination_right',
                             'arm_12_left',
                             'arm_12_right',
                             'arm_23_left',
                             'arm_23_right',
                             'leg_12_left',
                             'leg_12_right',
                             'leg_23_left',
                             'leg_23_right',
                             'arm_123_left',
                             'arm_123_right',
                             'leg_123_left',
                             'leg_123_right'], inplace = True)

# rename columns to accurate anatomical terms
select_pheno.rename(columns = {"hip_height": "pelvic_height",
                               "iliac_spine_left2iliac_spine_right": "pelvic_width",
                               "sciatic_notch_left2sciatic_notch_right": "pelvic_inlet_width",
                               "sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right": "iliac_flare_ratio",
                               "sacrum2pubic_tubercle": "oblique_pelvic_inlet_length",
                               "sciatic_notch2inferior_iliac_spine": "iliac_isthmus_breadth",
                               "iliopubic_eminence2acetabular_inferior": "acetabular_diameter",
                               "angle_pubic_tubercle_iliac_spine_lr": "iliac_flare_angle",
                               "pubic_arch_angle": "subpubic_angle",
                               "arm_12_length": "humerus",
                               "arm_23_length": "forearm",
                               "leg_12_length": "femur",
                               "leg_23_length": "tibia",
                               "ear_left2ear_right": "head_diameter",
                               "trochanter_left2trochanter_right": "trochanter_distance"}, inplace = True)

select_pheno['pelvic_inlet_area'] = (select_pheno['pelvic_inlet_width'] / 2) * (select_pheno['oblique_pelvic_inlet_length'] / 2) * math.pi

In [ ]:
select_pheno.columns

In [ ]:
select_pheno

In [ ]:
select_pheno.to_csv('key_results/hip_select_pheno_cm.csv', index=False)

In [ ]:
select_pheno

### filter with z-score (filter z here seems not approprate, should do this after filter eids)

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno_cm.csv'); hip_pheno

In [ ]:
hip_pheno_flt = hip_pheno[(np.abs(stats.zscore(hip_pheno.iloc[:, 8:])) < 4).all(axis=1)]; hip_pheno_flt

In [ ]:
hip_pheno_flt.to_csv('key_results/hip_select_pheno_flt.csv', index=False)

### filter with z-score for male and female

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_select_pheno.csv'); hip_pheno

In [ ]:
hip_pheno_male = hip_pheno[hip_pheno['sex'] == 1]
hip_pheno_male_flt = hip_pheno_male[(np.abs(stats.zscore(hip_pheno_male.iloc[:, 8:])) < 4).all(axis=1)]

hip_pheno_female = hip_pheno[hip_pheno['sex'] == 0]
hip_pheno_female_flt = hip_pheno_female[(np.abs(stats.zscore(hip_pheno_female.iloc[:, 8:])) < 4).all(axis=1)]

print(f"Male df shape before filter: {hip_pheno_male.shape}, after filter: {hip_pheno_male_flt.shape}")
print(f"Feale df shape before filter: {hip_pheno_female.shape}, after filter: {hip_pheno_female_flt.shape}")

In [ ]:
# save filtered data for both male and female
hip_pheno_male_flt.to_csv('key_results/hip_select_pheno_male_flt.csv', index=False)
hip_pheno_female_flt.to_csv('key_results/hip_select_pheno_female_flt.csv', index=False)

### Compare male and female phenotypes

In [ ]:
hip_pheno_male_flt = pd.read_csv('key_results/hip_select_pheno_male_flt.csv')
hip_pheno_female_flt = pd.read_csv('key_results/hip_select_pheno_female_flt.csv')

In [ ]:
hip_pheno_male_flt

In [ ]:
hip_pheno_female_flt

In [ ]:
hip_pheno_both_flt = pd.concat([hip_pheno_male_flt, hip_pheno_female_flt], axis=0); hip_pheno_both_flt

In [ ]:
# melt the dataframe to long format

hip_pheno_both_flt_melt = hip_pheno_both_flt.melt(id_vars = ['eid', 'sex'], 
                                                  value_vars=hip_pheno_both_flt.iloc[:, 4:].columns, 
                                                  var_name='phenotype', 
                                                  value_name='value'); hip_pheno_both_flt_melt

In [ ]:
hip_pheno_both_flt.columns

In [ ]:
hip_pheno_both_flt_melt.to_csv('key_results/hip_select_pheno_both_flt_melt.csv', index=False)

## All left and right correlation

In [ ]:
hip_pheno_flt = pd.read_csv('key_results/hip_select_pheno_flt.csv'); hip_pheno_flt

In [ ]:
plt.figure(figsize=(4, 4))
sns.regplot(x='acetabular_inclination_left', y='acetabular_inclination_right', data=hip_pheno_flt, 
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'}, 
            line_kws={'color': 'black', 'linewidth': 2})
r = pearsonr(hip_pheno_flt['acetabular_inclination_left'], hip_pheno_flt['acetabular_inclination_right'])[0]
plt.title('$r = {:.2f}$'.format(r))

# Make the plot a square
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim(38, 70)
plt.ylim(38, 70)
plt.xlabel('Left acetabular inclination (degree)')
plt.ylabel('Right acetabular inclination (degree)')
plt.tight_layout()
plt.savefig('out_fig/lvsr_acetabular_inclination.pdf', bbox_inches='tight')

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']
left_regions = [i for i in hip_regions if 'right' not in i]
lr_pairs = []
for i in range(len(left_regions)):
    for j in range(i + 1, len(left_regions)):
        l_line = f"{left_regions[i]}2{left_regions[j]}"
        if "left" in l_line: # remove keypoints all from central hip, i.e. only focusing on left
            r_line = l_line.replace("left", "right")
            lr_pair = [l_line, r_line]
            lr_pairs.append(lr_pair)

In [ ]:
len(lr_pairs)

In [ ]:
sns.set(style="ticks", font_scale=1.2)
fig, ax = plt.subplots(11, 7, figsize = (28, 45))

r_results = {}

for i in range(11):
    for j in range(7):
        idx = i * 7 + j
        if idx >= len(lr_pairs):
            ax[i, j].axis('off')
            continue
        lr_pair = lr_pairs[idx]
        sns.regplot(x = hip_pheno_flt[lr_pair[0]],
                    y = hip_pheno_flt[lr_pair[1]],
                    line_kws = {'color': 'black', "lw": 1},
                    scatter_kws = {'marker': 'o', 's': 20, 'alpha': 0.1},
                    ax = ax[i, j])
        
        r = pearsonr(hip_pheno_flt[lr_pair[0]], hip_pheno_flt[lr_pair[1]])[0]
        title = lr_pair[0].replace("_left", "").strip()

        r_results[lr_pair[0]] = r

        n = '\n'
        ax[i, j].set_title(f"{lr_pair[0]} {n} r = {r:.5f}")
        ax[i, j].spines['right'].set_visible(False)
        ax[i, j].spines['top'].set_visible(False)
        ax[i, j].set_ylabel("")
        ax[i, j].set_xlabel("")
        # ax[i, j].set_aspect('equal')
fig.text(0.5, -0.01, "Left side measurements (length:height)", ha='center', va='center', fontsize=20)
fig.text(-0.01, 0.5, "Right side measurements (length:height)", ha='center', va='center', rotation='vertical', fontsize=20)

plt.tight_layout()

plt.savefig("out_fig/left_vs_right_all_z_filter_23.pdf", bbox_inches = "tight")

In [ ]:
r_results

In [ ]:
r_results = pd.DataFrame.from_dict(r_results, orient='index', columns=['r'])

r_results.sort_values(by='r', ascending=False, inplace=True)
plt.figure(figsize=(5, 13))
ax = sns.barplot(x = 'r', y = r_results.index, data = r_results, color = 'steelblue')

# List of x-ticks to be colored red
yticks_to_color_red = ['sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

# Set y-tick labels color
for ytick in ax.get_yticklabels():
    if any(substring in ytick.get_text() for substring in yticks_to_color_red):
        ytick.set_color('red')

plt.xlabel('$r$')
plt.xlim(0, 1)
plt.yticks(fontsize=9)
plt.savefig("out_fig/left_vs_right_flt_r.pdf", bbox_inches = 'tight');

In [ ]:
r_results.to_csv('key_results/left_vs_right_flt_r.csv')

## selected phenotyps left and right correlation

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23_norm_height.csv'); hip_pheno

In [ ]:
select_pheno = hip_pheno[['image_id',
                          'eid',
                          'sex',
                          'file_name',
                          'hip_height',
                          'iliac_spine_left2iliac_spine_right',
                          'sciatic_notch_left2sciatic_notch_right',
                          'sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right',
                          'sacrum2pubic_tubercle',
                          'sciatic_notch_left2inferior_iliac_spine_left',
                          'sciatic_notch_right2inferior_iliac_spine_right',
                          'iliopubic_eminence_left2acetabular_inferior_left',
                          'iliopubic_eminence_right2acetabular_inferior_right',
                          'angle_pubic_tubercle_iliac_spine_lr',
                          'pubic_arch_angle',
                          'acetabular_inclination_left',
                          'acetabular_inclination_right',
                          'ear_left2ear_right',
                          'trochanter_left2trochanter_right']]

# filter with z
select_pheno_flt = select_pheno[(np.abs(stats.zscore(select_pheno.iloc[:, 4:])) < 4).all(axis=1)]; select_pheno_flt

In [ ]:
eids = pd.read_csv("key_results/eids_from_emily.csv")['eid'].tolist()
select_pheno_flt_white = select_pheno_flt[select_pheno_flt['eid'].isin(eids)]
select_pheno_flt_white.to_csv('key_results/hip_pheno_flt_white_for_corr_plot.csv', index=False); select_pheno_flt_white

### Plot

In [ ]:
df1 = select_pheno_flt_white[['sciatic_notch_left2inferior_iliac_spine_left', 'sciatic_notch_right2inferior_iliac_spine_right']]
df1['Group'] = 'Iliac ismthmus breadth'
df1.rename(columns={'sciatic_notch_left2inferior_iliac_spine_left': 'Left', 'sciatic_notch_right2inferior_iliac_spine_right': 'Right'}, inplace=True)

df2 = select_pheno_flt_white[['iliopubic_eminence_left2acetabular_inferior_left', 'iliopubic_eminence_right2acetabular_inferior_right']]
df2['Group'] = 'Acetabular diameter'
df2.rename(columns={'iliopubic_eminence_left2acetabular_inferior_left': 'Left', 'iliopubic_eminence_right2acetabular_inferior_right': 'Right'}, inplace=True)

df3 = select_pheno_flt_white[['acetabular_inclination_left', 'acetabular_inclination_right']]
df3['Group'] = 'Acetabular inclination'
df3.rename(columns={'acetabular_inclination_left': 'Left', 'acetabular_inclination_right': 'Right'}, inplace=True)

df = pd.concat([df1, df2, df3], axis=0)

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 15))

#########################################################
#             scatter plot for left vs right            #
#########################################################
sns.scatterplot(x='Left', y='Right', data=df1, alpha=0.5, s=20, ax=ax[0, 0], color=sns.color_palette("Paired")[1])
ax[0, 0].set_xlabel('Left side measurements (length:height)')
ax[0, 0].set_ylabel('Right side measurements (length:height)')
ax[0, 0].set_xlim(min(df1['Left'].min(), df1['Right'].min()) - 0.001, max(df1['Left'].max(), df1['Right'].max()) + 0.001)
ax[0, 0].set_ylim(min(df1['Left'].min(), df1['Right'].min()) - 0.001, max(df1['Left'].max(), df1['Right'].max()) + 0.001)
r = pearsonr(df1['Left'], df1['Right'])[0]
ax[0, 0].set_title('Iliac ismthmus breadth\n$r$ = {:.2f}'.format(r))
ax[0, 0].plot([0, 1], [0, 1], ls='--', c='black')

sns.scatterplot(x='Left', y='Right', data=df2, alpha=0.5, s=20, ax=ax[0, 1], color=sns.color_palette("Paired")[3])
ax[0, 1].set_xlabel('Left side measurements (length:height)')
ax[0, 1].set_ylabel('')
ax[0, 1].set_xlim(min(df2['Left'].min(), df2['Right'].min()) - 0.001, max(df2['Left'].max(), df2['Right'].max()) + 0.001)
ax[0, 1].set_ylim(min(df2['Left'].min(), df2['Right'].min()) - 0.001, max(df2['Left'].max(), df2['Right'].max()) + 0.001)
r = pearsonr(df2['Left'], df2['Right'])[0]
ax[0, 1].set_title('Acetabular diameter\n$r$ = {:.2f}'.format(r))
ax[0, 1].plot([0, 1], [0, 1], ls='--', c='black')

sns.scatterplot(x='Left', y='Right', data=df3, alpha=0.5, s=20, ax=ax[0, 2], color=sns.color_palette("Paired")[5])
ax[0, 2].set_xlabel('Left side measurements (degree)')
ax[0, 2].set_ylabel('Right side measurements (degree)')
ax[0, 2].set_xlim(min(df3['Left'].min(), df3['Right'].min()) - 1, max(df3['Left'].max(), df3['Right'].max()) + 1)
ax[0, 2].set_ylim(min(df3['Left'].min(), df3['Right'].min()) - 1, max(df3['Left'].max(), df3['Right'].max()) + 1)
r = pearsonr(df3['Left'], df3['Right'])[0]
ax[0, 2].set_title('Acetabular inclination\n$r$ = {:.2f}'.format(r))
ax[0, 2].plot([0, 100], [0, 100], ls='--', c='black')

#########################################################
#                        viloin plot                    #
#########################################################
df1_melt = df1.melt(id_vars=['Group'], value_vars=['Left', 'Right'])
colors = {'Left': sns.color_palette("Paired")[0], 'Right': sns.color_palette("Paired")[1]}
sns.violinplot(x='variable', y='value', data=df1_melt, ax=ax[1, 0], palette=colors, inner='box')

ax[1, 0].set_xlabel('')
ax[1, 0].set_ylabel('Phenotype (length:height)')

df2_melt = df2.melt(id_vars=['Group'], value_vars=['Left', 'Right'])
colors = {'Left': sns.color_palette("Paired")[2], 'Right': sns.color_palette("Paired")[3]}
sns.violinplot(x='variable', y='value', data=df2_melt, ax=ax[1, 1], palette=colors, inner='box')
ax[1, 1].set_xlabel('')
ax[1, 1].set_ylabel('')

df3_melt = df3.melt(id_vars=['Group'], value_vars=['Left', 'Right'])
colors = {'Left': sns.color_palette("Paired")[4], 'Right': sns.color_palette("Paired")[5]}
sns.violinplot(x='variable', y='value', data=df3_melt, ax=ax[1, 2], palette=colors, inner='box')
ax[1, 2].set_xlabel('')
ax[1, 2].set_ylabel('Phenotype (degree)')

#########################################################
# scatter plot for left right discrepancy in two visits #
#########################################################
visit_1_df = pd.read_csv('key_results/visit_1_df.csv')
visit_2_df = pd.read_csv('key_results/visit_2_df.csv')

visit_1_df['acetabular_diameter_discrepancy'] = visit_1_df['iliopubic_eminence_left2acetabular_inferior_left'] - visit_1_df['iliopubic_eminence_right2acetabular_inferior_right']
visit_1_df['acetabular_inclination_discrepancy'] = visit_1_df['acetabular_inclination_left'] - visit_1_df['acetabular_inclination_right']
visit_1_df['iliac_ismthmus_breadth'] = visit_1_df['sciatic_notch_left2inferior_iliac_spine_left'] - visit_1_df['sciatic_notch_right2inferior_iliac_spine_right']

visit_2_df['acetabular_diameter_discrepancy'] = visit_2_df['iliopubic_eminence_left2acetabular_inferior_left'] - visit_2_df['iliopubic_eminence_right2acetabular_inferior_right']
visit_2_df['acetabular_inclination_discrepancy'] = visit_2_df['acetabular_inclination_left'] - visit_2_df['acetabular_inclination_right']
visit_2_df['iliac_ismthmus_breadth'] = visit_2_df['sciatic_notch_left2inferior_iliac_spine_left'] - visit_2_df['sciatic_notch_right2inferior_iliac_spine_right']

two_visit_discrepancy = pd.merge(visit_1_df, visit_2_df, on='eid', suffixes=('_visit_1', '_visit_2'))

sns.scatterplot(x='iliac_ismthmus_breadth_visit_1', y='iliac_ismthmus_breadth_visit_2', data=two_visit_discrepancy, alpha=0.5, s=20, ax=ax[2, 0], color=sns.color_palette("Paired")[1])
ax[2, 0].set_xlabel('First visit discrepancy (length:height)')
ax[2, 0].set_ylabel('Second visit discrepancy (length:height)')
ax[2, 0].set_xlim(min(two_visit_discrepancy['iliac_ismthmus_breadth_visit_1'].min(), two_visit_discrepancy['iliac_ismthmus_breadth_visit_2'].min()) - 0.001, 
                  max(two_visit_discrepancy['iliac_ismthmus_breadth_visit_1'].max(), two_visit_discrepancy['iliac_ismthmus_breadth_visit_2'].max()) + 0.001)
ax[2, 0].set_ylim(min(two_visit_discrepancy['iliac_ismthmus_breadth_visit_1'].min(), two_visit_discrepancy['iliac_ismthmus_breadth_visit_2'].min()) - 0.001, 
                  max(two_visit_discrepancy['iliac_ismthmus_breadth_visit_1'].max(), two_visit_discrepancy['iliac_ismthmus_breadth_visit_2'].max()) + 0.001)
r = pearsonr(two_visit_discrepancy['iliac_ismthmus_breadth_visit_1'], two_visit_discrepancy['iliac_ismthmus_breadth_visit_2'])[0]
ax[2, 0].set_title('$r$ = {:.2f}'.format(r))
ax[2, 0].plot([-1, 1], [-1, 1], ls='--', c='black')

sns.scatterplot(x='acetabular_diameter_discrepancy_visit_1', y='acetabular_diameter_discrepancy_visit_2', data=two_visit_discrepancy, alpha=0.5, s=20, ax=ax[2, 1], color=sns.color_palette("Paired")[3])
ax[2, 1].set_xlabel('First visit discrepancy (length:height)')
ax[2, 1].set_ylabel('')
ax[2, 1].set_xlim(min(two_visit_discrepancy['acetabular_diameter_discrepancy_visit_1'].min(), two_visit_discrepancy['acetabular_diameter_discrepancy_visit_2'].min()) - 0.001, 
                  max(two_visit_discrepancy['acetabular_diameter_discrepancy_visit_1'].max(), two_visit_discrepancy['acetabular_diameter_discrepancy_visit_2'].max()) + 0.001)
ax[2, 1].set_ylim(min(two_visit_discrepancy['acetabular_diameter_discrepancy_visit_1'].min(), two_visit_discrepancy['acetabular_diameter_discrepancy_visit_2'].min()) - 0.001, 
                  max(two_visit_discrepancy['acetabular_diameter_discrepancy_visit_1'].max(), two_visit_discrepancy['acetabular_diameter_discrepancy_visit_2'].max()) + 0.001)
r = pearsonr(two_visit_discrepancy['acetabular_diameter_discrepancy_visit_1'], two_visit_discrepancy['acetabular_diameter_discrepancy_visit_2'])[0]
ax[2, 1].set_title('$r$ = {:.2f}'.format(r))
ax[2, 1].plot([-1, 1], [-1, 1], ls='--', c='black')

sns.scatterplot(x='acetabular_inclination_discrepancy_visit_1', y='acetabular_inclination_discrepancy_visit_2', data=two_visit_discrepancy, alpha=0.5, s=20, ax=ax[2, 2], color=sns.color_palette("Paired")[5])
ax[2, 2].set_xlabel('First visit discrepancy (degree)')
ax[2, 2].set_ylabel('Second visit discrepancy (degree)')
ax[2, 2].set_xlim(min(two_visit_discrepancy['acetabular_inclination_discrepancy_visit_1'].min(), two_visit_discrepancy['acetabular_inclination_discrepancy_visit_2'].min()) - 1, 
                  max(two_visit_discrepancy['acetabular_inclination_discrepancy_visit_1'].max(), two_visit_discrepancy['acetabular_inclination_discrepancy_visit_2'].max()) + 1)
ax[2, 2].set_ylim(min(two_visit_discrepancy['acetabular_inclination_discrepancy_visit_1'].min(), two_visit_discrepancy['acetabular_inclination_discrepancy_visit_2'].min()) - 1, 
                  max(two_visit_discrepancy['acetabular_inclination_discrepancy_visit_1'].max(), two_visit_discrepancy['acetabular_inclination_discrepancy_visit_2'].max()) + 1)
r = pearsonr(two_visit_discrepancy['acetabular_inclination_discrepancy_visit_1'], two_visit_discrepancy['acetabular_inclination_discrepancy_visit_2'])[0]
ax[2, 2].set_title('$r$ = {:.2f}'.format(r))
ax[2, 2].plot([-100, 100], [-100, 100], ls='--', c='black')

plt.tight_layout()
plt.savefig('out_fig/select_left_vs_right.pdf', bbox_inches='tight')

### Compare Eucharist model and mine by seeing if there are any discrepancy correlation between two visits

In [ ]:
visit_1_df = pd.read_csv('key_results/visit_1_df.csv')
visit_2_df = pd.read_csv('key_results/visit_2_df.csv')

ek_df_82 = pd.read_csv("../from_eucharist/864_patient_distances.csv", sep = '\t')
ek_df_93 = pd.read_csv("../from_eucharist/960_patient_distances.csv", sep = '\t')
ek_df_82['size'] = '816x288'
ek_df_93['size'] = '960x384'
ek_df_82['Image ID'] = ek_df_82['Image ID'].astype(str).str.zfill(5)
ek_df_82['Image ID'] = ek_df_82['Image ID'].apply(lambda x: x + '_82')
ek_df_93['Image ID'] = ek_df_93['Image ID'].astype(str).str.zfill(5)
ek_df_93['Image ID'] = ek_df_93['Image ID'].apply(lambda x: x + '_93')
ek_df = pd.concat([ek_df_82, ek_df_93])

# Eucharist data need to be convert to cm
img_beta_df = pd.read_csv('../UKB_xray_image_info/img_beta_df.csv')[['image_id', 'Value']]

ek_df = ek_df.merge(img_beta_df, left_on='Image ID', right_on='image_id', how='inner')
ek_df = ek_df.drop(columns=['image_id', 'size'])
ek_df.iloc[:, 4:] = ek_df[ek_df.columns[4:]].multiply(ek_df['Value'], axis="index")

ek_visit_1 = ek_df.merge(visit_1_df, left_on='Image ID', right_on='image_id')
ek_visit_2 = ek_df.merge(visit_2_df, left_on='Image ID', right_on='image_id')

ek_visit_1['arm_ratio_visit_1'] = (ek_visit_1['left_elbow-left_wrist'] + ek_visit_1['left_shoulder-left_elbow']) / (ek_visit_1['right_elbow-right_wrist'] + ek_visit_1['right_shoulder-right_elbow'])
ek_visit_2['arm_ratio_visit_2'] = (ek_visit_2['left_elbow-left_wrist'] + ek_visit_2['left_shoulder-left_elbow']) / (ek_visit_2['right_elbow-right_wrist'] + ek_visit_2['right_shoulder-right_elbow'])

ek_arm_ratio = pd.merge(ek_visit_1, ek_visit_2, on='Patient EID', how='inner')[['Patient EID', 'arm_ratio_visit_1', 'arm_ratio_visit_2']]

# filter with z
ek_arm_ratio_flt = ek_arm_ratio[(np.abs(stats.zscore(ek_arm_ratio.iloc[:, 1:])) < 4).all(axis=1)]
ek_arm_ratio_flt.to_csv('key_results/ek_arm_left_right_ratio_flt.csv', index=False)

In [ ]:
r, p = pearsonr(ek_arm_ratio_flt['arm_ratio_visit_1'], ek_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = ek_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

In [ ]:
me_df = pd.read_csv("../left_vs_right/pred_res/pheno_cm_length.csv", index_col=0)
me_visit_1 = me_df.merge(visit_1_df, left_index=True, right_on='image_id')
me_visit_2 = me_df.merge(visit_2_df, left_index=True, right_on='image_id')

me_visit_1['arm_ratio_visit_1'] = me_visit_1['arm_123_left'] / me_visit_1['arm_123_right']
me_visit_2['arm_ratio_visit_2'] = me_visit_2['arm_123_left'] / me_visit_2['arm_123_right']

me_arm_ratio = pd.merge(me_visit_1, me_visit_2, on='eid', how='inner')[['eid', 'arm_ratio_visit_1', 'arm_ratio_visit_2']]

# filter with z
me_arm_ratio_flt = me_arm_ratio[(np.abs(stats.zscore(me_arm_ratio.iloc[:, 1:])) < 4).all(axis=1)]
me_arm_ratio_flt.to_csv("key_results/me_arm_left_right_ratio_flt.csv")

In [ ]:
r, p = pearsonr(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = me_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

### Train the HRNet for multiple times

In [ ]:
# load prediction
train1 = json.load(open('../left_vs_right/pred_res/arm_kps_pred_1.json'))
train2 = json.load(open('../left_vs_right/pred_res/arm_kps_pred_2.json'))
train3 = json.load(open('../left_vs_right/pred_res/arm_kps_pred_3.json'))

# load two visit data
visit_1_df = pd.read_csv('key_results/visit_1_df.csv')
visit_2_df = pd.read_csv('key_results/visit_2_df.csv')

In [ ]:
train3

In [ ]:
def calc_dist(pt1, pt2):
    return np.sqrt((pt1[0] - pt2[0])**2 + (pt1[1] - pt2[1])**2)

In [ ]:
def calc_r2_two_visit(json_file, num_train):
    # read json file
    j_file = json.load(open(json_file))

    # make a dataframe for pixel length
    arm_length = ['arm_12_left', 'arm_12_right', 'arm_23_left', 'arm_23_right', 'arm_123_left', 'arm_123_right']
    idx = list(j_file.keys())
    arm_length_df = pd.DataFrame(index=idx, columns=arm_length)

    # calculate arm length (pixel)
    for i in idx:
        arm_length_df.loc[i, 'arm_12_left'] = calc_dist(j_file[i][0], j_file[i][2])
        arm_length_df.loc[i, 'arm_12_right'] = calc_dist(j_file[i][1], j_file[i][3])
        arm_length_df.loc[i, 'arm_23_left'] = calc_dist(j_file[i][2], j_file[i][4])
        arm_length_df.loc[i, 'arm_23_right'] = calc_dist(j_file[i][3], j_file[i][5])
        arm_length_df.loc[i, 'arm_123_left'] = calc_dist(j_file[i][0], j_file[i][2]) + calc_dist(j_file[i][2], j_file[i][4])
        arm_length_df.loc[i, 'arm_123_right'] = calc_dist(j_file[i][1], j_file[i][3]) + calc_dist(j_file[i][3], j_file[i][5])
    
    arm_length_df = arm_length_df.astype(float)
    
    # convert pixel to cm
    img_beta_df = pd.read_csv('../UKB_xray_image_info/img_beta_df.csv')[['image_id', 'Value']]
    arm_length_df = arm_length_df.merge(img_beta_df, left_index=True, right_on='image_id', how='inner')
    arm_length_df.iloc[:, :-2] = arm_length_df.iloc[:, :-2].multiply(arm_length_df['Value'], axis="index")

    arm_length_df.set_index('image_id', inplace=True)
    arm_length_df.drop(columns=['Value'], inplace=True)

    # save arm length dataframe
    # arm_length_df.to_csv(f'key_results/arm_length_{num_train}.csv')

    # divide to visit 1 and visit 2 dataframe
    me_visit_1 = arm_length_df.merge(visit_1_df, left_index=True, right_on='image_id')
    me_visit_2 = arm_length_df.merge(visit_2_df, left_index=True, right_on='image_id')

    me_visit_1['arm_ratio_visit_1'] = me_visit_1['arm_123_left'] / me_visit_1['arm_123_right']
    me_visit_2['arm_ratio_visit_2'] = me_visit_2['arm_123_left'] / me_visit_2['arm_123_right']

    me_arm_ratio = pd.merge(me_visit_1, me_visit_2, on='eid', how='inner')[['eid', 'arm_ratio_visit_1', 'arm_ratio_visit_2']]

    # filter with z
    me_arm_ratio_flt = me_arm_ratio[(np.abs(stats.zscore(me_arm_ratio.iloc[:, 1:])) < 4).all(axis=1)]
    # me_arm_ratio_flt.to_csv(f"key_results/arm_length_ratio_flt_{num_train}.csv")
    
    return r2_score(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])


In [ ]:
calc_r2_two_visit('../left_vs_right/pred_res/arm_kps_pred.json', 1)

In [ ]:
calc_r2_two_visit('../left_vs_right/pred_res/arm_kps_ppred.json', 1)

In [ ]:
# read json file
j_file = json.load(open('../left_vs_right/pred_res/arm_kps_pred_4.json'))

# make a dataframe for pixel length
arm_length = ['arm_12_left', 'arm_12_right', 'arm_23_left', 'arm_23_right', 'arm_123_left', 'arm_123_right']
idx = list(j_file.keys())
arm_length_df = pd.DataFrame(index=idx, columns=arm_length)

# calculate arm length (pixel)
for i in idx:
    arm_length_df.loc[i, 'arm_12_left'] = calc_dist(j_file[i][0], j_file[i][2])
    arm_length_df.loc[i, 'arm_12_right'] = calc_dist(j_file[i][1], j_file[i][3])
    arm_length_df.loc[i, 'arm_23_left'] = calc_dist(j_file[i][2], j_file[i][4])
    arm_length_df.loc[i, 'arm_23_right'] = calc_dist(j_file[i][3], j_file[i][5])
    arm_length_df.loc[i, 'arm_123_left'] = calc_dist(j_file[i][0], j_file[i][2]) + calc_dist(j_file[i][2], j_file[i][4])
    arm_length_df.loc[i, 'arm_123_right'] = calc_dist(j_file[i][1], j_file[i][3]) + calc_dist(j_file[i][3], j_file[i][5])

arm_length_df = arm_length_df.astype(float)

# convert pixel to cm
img_beta_df = pd.read_csv('../UKB_xray_image_info/img_beta_df.csv')[['image_id', 'Value']]
arm_length_df = arm_length_df.merge(img_beta_df, left_index=True, right_on='image_id', how='inner')
arm_length_df.iloc[:, :-2] = arm_length_df.iloc[:, :-2].multiply(arm_length_df['Value'], axis="index")

arm_length_df.set_index('image_id', inplace=True)
arm_length_df.drop(columns=['Value'], inplace=True)

# save arm length dataframe
# arm_length_df.to_csv(f'key_results/arm_length_{num_train}.csv')

# divide to visit 1 and visit 2 dataframe
me_visit_1 = arm_length_df.merge(visit_1_df, left_index=True, right_on='image_id')
me_visit_2 = arm_length_df.merge(visit_2_df, left_index=True, right_on='image_id')

me_visit_1['arm_ratio_visit_1'] = me_visit_1['arm_123_left'] / me_visit_1['arm_123_right']
me_visit_2['arm_ratio_visit_2'] = me_visit_2['arm_123_left'] / me_visit_2['arm_123_right']

me_arm_ratio = pd.merge(me_visit_1, me_visit_2, on='eid', how='inner')[['eid', 'arm_ratio_visit_1', 'arm_ratio_visit_2']]

# filter with z
me_arm_ratio_flt = me_arm_ratio[(np.abs(stats.zscore(me_arm_ratio.iloc[:, 1:])) < 4).all(axis=1)]
# me_arm_ratio_flt.to_csv(f"key_results/arm_length_ratio_flt_{num_train}.csv")

In [ ]:
r, p = pearsonr(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = me_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

In [ ]:
r, p = pearsonr(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = me_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

In [ ]:
r, p = pearsonr(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = me_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

In [ ]:
r, p = pearsonr(me_arm_ratio_flt['arm_ratio_visit_1'], me_arm_ratio_flt['arm_ratio_visit_2'])

sns.scatterplot(x = 'arm_ratio_visit_1', y = 'arm_ratio_visit_2', data = me_arm_ratio_flt)
plt.annotate(f'r = {r:.3f}\np-value = {p:.5f}', xy=(0.05, 0.95), xycoords='axes fraction', verticalalignment='top')

## correlation between 1st and 2nd visit

In [ ]:
hip_pheno_flt = pd.read_csv('key_results/hip_pheno_23_flt.csv')
# all_dcm_info = pd.read_csv("all_prediction/all_dcm_info.csv")[['image_id', 'file_name']]
# hip_pheno_flt = all_dcm_info.merge(hip_pheno_flt, on='image_id'); hip_pheno_flt
hip_pheno_flt

In [ ]:
p_eid_count = Counter(hip_pheno_flt['Patient EID'].tolist())

In [ ]:
two_dup = [key for key, value in p_eid_count.items() if value > 1]
three_dup = [key for key, value in p_eid_count.items() if value > 2]
only_two_dup = [i for i in two_dup if i not in three_dup]

In [ ]:
two_visit_p = hip_pheno_flt[hip_pheno_flt['Patient EID'].isin(only_two_dup)]

In [ ]:
two_visit_p = two_visit_p.sort_values(by = 'file_name'); two_visit_p

In [ ]:
p_eids = []
visit_1_fn = []
visit_2_fn = []

for i in two_visit_p.index:
    p_eid = two_visit_p.loc[i, 'Patient EID']
    fn = two_visit_p.loc[i, 'file_name']
    if p_eid in p_eids:
        visit_2_fn.append(fn)
    else:
        p_eids.append(p_eid)
        visit_1_fn.append(fn)

In [ ]:
visit_1_df = two_visit_p[two_visit_p['file_name'].isin(visit_1_fn)]
visit_2_df = two_visit_p[two_visit_p['file_name'].isin(visit_2_fn)]

In [ ]:
visit_1_df = visit_1_df.sort_values(by = 'Patient EID')
visit_2_df = visit_2_df.sort_values(by = 'Patient EID')

In [ ]:
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

In [ ]:
visit_1_df

In [ ]:
visit_2_df

#### Two visits for angles

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

sns.regplot(x=visit_1_df['acetabular_inclination_left'], y=visit_2_df['acetabular_inclination_left'],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'}, 
            line_kws={'color': 'black', 'linewidth': 2}, 
            ax=ax[0])
r = pearsonr(visit_1_df['acetabular_inclination_left'], visit_2_df['acetabular_inclination_left'])[0]
ax[0].set_title('Acetabular inclination left \n $r = {:.3f}$'.format(r))
ax[0].set_aspect('equal', adjustable='box')
ax[0].set_xlabel('')
ax[0].set_ylabel('Second visit measurement')

sns.regplot(x=visit_1_df['acetabular_inclination_right'], y=visit_2_df['acetabular_inclination_right'],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'}, 
            line_kws={'color': 'black', 'linewidth': 2}, 
            ax=ax[1])
r = pearsonr(visit_1_df['acetabular_inclination_right'], visit_2_df['acetabular_inclination_right'])[0]
ax[1].set_title('Acetabular inclination right \n $r = {:.3f}$'.format(r))
ax[1].set_aspect('equal', adjustable='box')
ax[1].set_xlabel('First visit measurement')
ax[1].set_ylabel('')

sns.regplot(x=visit_1_df['pubic_arch_angle'], y=visit_2_df['pubic_arch_angle'],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'}, 
            line_kws={'color': 'black', 'linewidth': 2}, 
            ax=ax[2])
r = pearsonr(visit_1_df['pubic_arch_angle'], visit_2_df['pubic_arch_angle'])[0]
ax[2].set_title('Pubic arch angle \n $r = {:.3f}$'.format(r))
ax[2].set_aspect('equal', adjustable='box')
ax[2].set_xlabel('')
ax[2].set_ylabel('')

plt.tight_layout()

plt.savefig('out_fig/angles_first_vs_second_visit.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

sns.regplot(x=visit_1_df['acetabular_inclination_left'] - visit_1_df['acetabular_inclination_right'], 
            y=visit_2_df['acetabular_inclination_left'] - visit_2_df['acetabular_inclination_right'],
            scatter_kws={'alpha': 0.3, 'color': 'steelblue'}, 
            line_kws={'color': 'black', 'linewidth': 2})
r = pearsonr(visit_1_df['acetabular_inclination_left'] - visit_1_df['acetabular_inclination_right'], 
             visit_2_df['acetabular_inclination_left'] - visit_2_df['acetabular_inclination_right'])[0]
ax.set_title('Acetabular inclination different \n $r = {:.3f}$'.format(r))
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('First visit measurement')
ax.set_ylabel('Second visit measurement')
ax.set_xlim(-16, 16)
ax.set_ylim(-16, 16)
plt.savefig('results/acetabular_inclination_diff.pdf', bbox_inches='tight')


#### Two visits for length 

In [ ]:
two_visit_dict = {}
for i in range(len(hip_regions)):
    for j in range(i+1, len(hip_regions)):
        visit1 = visit_1_df[hip_regions[i] + '2' + hip_regions[j]].tolist()
        visit2 = visit_2_df[hip_regions[i] + '2' + hip_regions[j]].tolist()
        r = pearsonr(visit1, visit2)[0]
        two_visit_dict[hip_regions[i] + '2' + hip_regions[j]] = r

In [ ]:
r = pd.DataFrame.from_dict(two_visit_dict, orient='index', columns=['r'])

In [ ]:
r = r.sort_values(by = 'r', ascending = False)

In [ ]:
r

In [ ]:
plt.figure(figsize=(5, 33))
ax = sns.barplot(x = 'r', y = r.index, data = r, color = 'steelblue')

# List of x-ticks to be colored red
yticks_to_color_red = ['sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

# Set y-tick labels color
for ytick in ax.get_yticklabels():
    if any(substring in ytick.get_text() for substring in yticks_to_color_red):
        ytick.set_color('red')

plt.xlabel('$r$')
plt.xlim(0, 1)
plt.yticks(fontsize=7)
plt.savefig("out_fig/two_visit_flt_r.pdf", bbox_inches = 'tight');

In [ ]:
r.to_csv('key_results/two_visit_flt_r.csv')

## select phenotypes 1st and 2nd correlation

In [ ]:
select_pheno = pd.read_csv('key_results/hip_select_pheno_cm.csv'); select_pheno # for left and right pheno

In [ ]:
select_pheno_flt = select_pheno[(np.abs(stats.zscore(select_pheno.iloc[:, 8:])) < 4).all(axis=1)]; select_pheno_flt

In [ ]:
# convert file name to datetime

from datetime import datetime

def convert_to_date(filename):
    try:
        return datetime.strptime(filename.split(".")[7], '%Y%m%d%H%M%S').strftime('%Y-%m-%d %H:%M:%S')
    except Exception as e:
        print(f"Weird time format for filename: {filename}. Error: {e}")
        return None

select_pheno_flt['file_name'] = select_pheno_flt['file_name'].apply(convert_to_date)

# Remove rows that couldn't be converted
select_pheno_flt = select_pheno_flt.dropna(subset=['file_name'])

In [ ]:
p_eid_count = Counter(select_pheno_flt['eid'].tolist())

two_dup = [key for key, value in p_eid_count.items() if value > 1]
three_dup = [key for key, value in p_eid_count.items() if value > 2]
only_two_dup = [i for i in two_dup if i not in three_dup]

# get eid only two visits
two_visit_p = select_pheno_flt[select_pheno_flt['eid'].isin(only_two_dup)]

two_visit_df = two_visit_p.copy()

two_visit_p['file_name'] = pd.to_datetime(two_visit_p['file_name'])
two_visit_p = two_visit_p.sort_values(by = ['eid', 'file_name'])

# Calculate the difference between consecutive dates for each eid
two_visit_p['date_diff'] = two_visit_p.groupby('eid')['file_name'].diff()

# Convert the difference to years
two_visit_p['date_diff'] = two_visit_p['date_diff'].dt.total_seconds() / (60*60*24*365)

# Filter out rows where the difference is less than 2 years
two_visit_p = two_visit_p[two_visit_p['date_diff'] >= 2]

eids_2year_apart = two_visit_p['eid'].tolist()

# get qualified eid
two_visit_df = two_visit_df[two_visit_df['eid'].isin(eids_2year_apart)]

In [ ]:
p_eids = []
visit_1_fn = []
visit_2_fn = []

two_visit_df.sort_values(by = ['eid', 'file_name'], inplace = True)
two_visit_df.reset_index(drop = True, inplace = True)

for i in two_visit_df.index:
    p_eid = two_visit_df.loc[i, 'eid']
    fn = two_visit_df.loc[i, 'file_name']
    if p_eid in p_eids:
        visit_2_fn.append(fn)
    else:
        p_eids.append(p_eid)
        visit_1_fn.append(fn)

visit_1_df = two_visit_df[two_visit_df['file_name'].isin(visit_1_fn)]
visit_2_df = two_visit_df[two_visit_df['file_name'].isin(visit_2_fn)]

visit_1_df = visit_1_df.sort_values(by = 'eid')
visit_2_df = visit_2_df.sort_values(by = 'eid')

In [ ]:
visit_1_df.shape

In [ ]:
# visit_1_df.to_csv('key_results/visit_1_df_avg_left_right.csv', index = False)
# visit_2_df.to_csv('key_results/visit_2_df_avg_left_right.csv', index = False)

visit_1_df.to_csv('key_results/visit_1_df.csv', index = False)
visit_2_df.to_csv('key_results/visit_2_df.csv', index = False)

## phenotypes left and right correlation

In [ ]:
hip_pheno = pd.read_csv('key_results/hip_pheno_23_cm.csv'); hip_pheno

In [ ]:
hip_pheno = hip_pheno[[
    'eid', 'standing_height', 'sex', 'age', 'bmi', 'weight', 'image_id', 'file_name',
    'sciatic_notch_left2inferior_iliac_spine_left',
    'sciatic_notch_right2inferior_iliac_spine_right',
    'iliopubic_eminence_left2acetabular_inferior_left',
    'iliopubic_eminence_right2acetabular_inferior_right',
    'acetabular_inclination_left', 'acetabular_inclination_right',
]]

hip_pheno_flt = hip_pheno[(np.abs(stats.zscore(hip_pheno.iloc[:, 8:])) < 4).all(axis=1)]

visit_1_imageid = pd.read_csv('key_results/visit_1_df.csv')['image_id'].tolist()
visit_2_imageid = pd.read_csv('key_results/visit_2_df.csv')['image_id'].tolist()

visit_1_df_w_lr = hip_pheno_flt[hip_pheno_flt['image_id'].isin(visit_1_imageid)]
visit_2_df_w_lr = hip_pheno_flt[hip_pheno_flt['image_id'].isin(visit_2_imageid)]

visit_1_df_w_lr.to_csv('key_results/visit_1_df_w_lr.csv', index = False)
visit_2_df_w_lr.to_csv('key_results/visit_2_df_w_lr.csv', index = False)

In [ ]:
print(visit_1_df_w_lr.shape)
print(visit_2_df_w_lr.shape)

### Plot

In [ ]:
visit_1_df = pd.read_csv('key_results/visit_1_df.csv')
visit_2_df = pd.read_csv('key_results/visit_2_df.csv')

In [ ]:
visit_1_df.drop(columns = ['image_id', 'sex', 'file_name'], inplace = True)
visit_2_df.drop(columns = ['image_id', 'sex', 'file_name'], inplace = True)

# calculate the left and right average

# visit 1
visit_1_df['sciatic_notch2inferior_iliac_spine'] = \
    (visit_1_df['sciatic_notch_left2inferior_iliac_spine_left'] + visit_1_df['sciatic_notch_right2inferior_iliac_spine_right']) / 2
visit_1_df['iliopubic_eminence2acetabular_inferior'] = \
    (visit_1_df['iliopubic_eminence_left2acetabular_inferior_left'] + visit_1_df['iliopubic_eminence_right2acetabular_inferior_right']) / 2
visit_1_df['acetabular_inclination'] = \
    (visit_1_df['acetabular_inclination_left'] + visit_1_df['acetabular_inclination_right']) / 2

visit_1_df.drop(columns = ['sciatic_notch_left2inferior_iliac_spine_left', 
                           'sciatic_notch_right2inferior_iliac_spine_right', 
                           'iliopubic_eminence_left2acetabular_inferior_left', 
                           'iliopubic_eminence_right2acetabular_inferior_right',
                           'acetabular_inclination_left',
                           'acetabular_inclination_right'], inplace = True)

# rename columns to accurate anatomical terms
visit_1_df.rename(columns = {"hip_height": "pelvic_height",
                             "ear_left2ear_right": "head_width",
                             "iliac_spine_left2iliac_spine_right": "pelvic_width",
                             "sciatic_notch_left2sciatic_notch_right": "pelvic_inlet_width",
                             "sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right": "iliac_flare_ratio",
                             "sacrum2pubic_tubercle": "oblique_pelvic_inlet_length",
                             "sciatic_notch2inferior_iliac_spine": "iliac_isthmus_breadth",
                             "iliopubic_eminence2acetabular_inferior": "acetabular_diameter",
                             "angle_pubic_tubercle_iliac_spine_lr": "iliac_flare_angle",
                             "pubic_arch_angle": "subpubic_angle",
                             "trochanter_left2trochanter_right": "trochanters_distance"}, inplace = True)

# visit 2
visit_2_df['sciatic_notch2inferior_iliac_spine'] = \
    (visit_2_df['sciatic_notch_left2inferior_iliac_spine_left'] + visit_2_df['sciatic_notch_right2inferior_iliac_spine_right']) / 2
visit_2_df['iliopubic_eminence2acetabular_inferior'] = \
    (visit_2_df['iliopubic_eminence_left2acetabular_inferior_left'] + visit_2_df['iliopubic_eminence_right2acetabular_inferior_right']) / 2
visit_2_df['acetabular_inclination'] = \
    (visit_2_df['acetabular_inclination_left'] + visit_2_df['acetabular_inclination_right']) / 2

visit_2_df.drop(columns = ['sciatic_notch_left2inferior_iliac_spine_left', 
                           'sciatic_notch_right2inferior_iliac_spine_right', 
                           'iliopubic_eminence_left2acetabular_inferior_left', 
                           'iliopubic_eminence_right2acetabular_inferior_right',
                           'acetabular_inclination_left',
                           'acetabular_inclination_right'], inplace = True)

# rename columns to accurate anatomical terms
visit_2_df.rename(columns = {"hip_height": "pelvic_height",
                             "ear_left2ear_right": "head_width",
                             "iliac_spine_left2iliac_spine_right": "pelvic_width",
                             "sciatic_notch_left2sciatic_notch_right": "pelvic_inlet_width",
                             "sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right": "iliac_flare_ratio",
                             "sacrum2pubic_tubercle": "oblique_pelvic_inlet_length",
                             "sciatic_notch2inferior_iliac_spine": "iliac_isthmus_breadth",
                             "iliopubic_eminence2acetabular_inferior": "acetabular_diameter",
                             "angle_pubic_tubercle_iliac_spine_lr": "iliac_flare_angle",
                             "pubic_arch_angle": "subpubic_angle",
                             "trochanter_left2trochanter_right": "trochanters_distance"}, inplace = True)


In [ ]:
visit_1_melt = pd.melt(visit_1_df, id_vars = ['eid'], var_name = 'pheno', value_name = 'value')
visit_2_melt = pd.melt(visit_2_df, id_vars = ['eid'], var_name = 'pheno', value_name = 'value')

visit_melt = pd.merge(visit_1_melt, visit_2_melt, on = ['eid', 'pheno'], suffixes = ('_visit_1', '_visit_2')); visit_melt

In [ ]:
length_pheno = ['pelvic_height', 'pelvic_width', 'pelvic_inlet_width',
                'oblique_pelvic_inlet_length', 'head_width',
                'trochanters_distance', 'iliac_isthmus_breadth',
                'acetabular_diameter']

angle_pheno = ['iliac_flare_angle', 'subpubic_angle', 'acetabular_inclination']

ratio_pheno = ['iliac_flare_ratio']

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (15, 5))

palette = sns.husl_palette(12, h=.8, l=.6)

# length phenotypes
visit_melt_length = visit_melt[visit_melt['pheno'].isin(length_pheno)]
# color_dict = {'pelvic_height': '#fdbf6f', 'pelvic_width': '#ff7f00', 'pelvic_inlet_width': '#6a3d9a', 'oblique_pelvic_inlet_length': '#b15928',
#               'head_width': '#cab2d6', 'trochanters_distance': '#d873c6', 'iliac_isthmus_breadth': '#1f78b4', 'acetabular_diameter': '#33a02c'}
color_dict = {'pelvic_height': palette[0], 'pelvic_width': palette[1], 'pelvic_inlet_width': palette[2], 'oblique_pelvic_inlet_length': palette[4],
              'head_width': palette[5], 'trochanters_distance': palette[6], 'iliac_isthmus_breadth': palette[3], 'acetabular_diameter': palette[7]}

sns.scatterplot(data = visit_melt_length, x = 'value_visit_1', y = 'value_visit_2', hue = 'pheno', palette = color_dict, ax = ax[0])
ax[0].plot([0, 1], [0, 1], transform = ax[0].transAxes, ls = '--', c = 'black')

leg = ax[0].legend(title="", loc='lower right', prop={'size': 8})
# Change the legend names. Replace 'new_label_1', 'new_label_2', etc. with your new names.
new_labels = ['Pelvic height', 'Pelvic width', 'Pelvic inlet width', 'Oblique pelvic inlet length', 'Head width', 'Trochanters distance', 'Iliac isthmus breadth', 'Acetabular diameter']
for t, l in zip(leg.texts[0:], new_labels):  # Skip the title, start from index 1
    t.set_text(l)

r = stats.pearsonr(visit_melt_length['value_visit_1'], visit_melt_length['value_visit_2'])[0]
ax[0].set_title("Length phenotypes\n$r$ = {:.5f}".format(r))
ax[0].set_xlabel('First visit (length:height)')
ax[0].set_ylabel('Second visit (length:height)')
ax[0].set_xlim(min(visit_melt_length['value_visit_1'].min(), visit_melt_length['value_visit_2'].min()) - 0.01, 
                  max(visit_melt_length['value_visit_1'].max(), visit_melt_length['value_visit_2'].max()) + 0.01)
ax[0].set_ylim(min(visit_melt_length['value_visit_1'].min(), visit_melt_length['value_visit_2'].min()) - 0.01, 
                  max(visit_melt_length['value_visit_1'].max(), visit_melt_length['value_visit_2'].max()) + 0.01)

# angle phenotypes
visit_melt_angle = visit_melt[visit_melt['pheno'].isin(angle_pheno)]
# color_dict = {'iliac_flare_angle': '#fdbf6f', 'subpubic_angle': '#ff7f00', 'acetabular_inclination': '#6a3d9a'}
color_dict = {'iliac_flare_angle': palette[8], 'subpubic_angle': palette[9], 'acetabular_inclination': palette[11]}

sns.scatterplot(data = visit_melt_angle, x = 'value_visit_1', y = 'value_visit_2', hue = 'pheno', palette = color_dict, ax = ax[1])
ax[1].plot([0, 180], [0, 180], transform = ax[1].transAxes, ls = '--', c = 'black')

leg = ax[1].legend(title="", loc='lower right', prop={'size': 8})
# Change the legend names. Replace 'new_label_1', 'new_label_2', etc. with your new names.
new_labels = ['Iliac flare angle', 'Subpubic angle', 'Acetabular inclination']
for t, l in zip(leg.texts[0:], new_labels):  # Skip the title, start from index 1
    t.set_text(l)

r = stats.pearsonr(visit_melt_angle['value_visit_1'], visit_melt_angle['value_visit_2'])[0]
ax[1].set_title("Angle phenotypes\n$r$ = {:.5f}".format(r))
ax[1].set_xlabel('First visit (degrees)')
ax[1].set_ylabel('Second visit (degrees)')
ax[1].set_xlim(min(visit_melt_angle['value_visit_1'].min(), visit_melt_angle['value_visit_2'].min()) - 10, 
               max(visit_melt_angle['value_visit_1'].max(), visit_melt_angle['value_visit_2'].max()) + 10)
ax[1].set_ylim(min(visit_melt_angle['value_visit_1'].min(), visit_melt_angle['value_visit_2'].min()) - 10, 
               max(visit_melt_angle['value_visit_1'].max(), visit_melt_angle['value_visit_2'].max()) + 10)

# ratio phenotypes
visit_melt_ratio = visit_melt[visit_melt['pheno'].isin(ratio_pheno)]

color_dict = {'iliac_flare_ratio': palette[10]}

sns.scatterplot(data = visit_melt_ratio, x = 'value_visit_1', y = 'value_visit_2', hue = 'pheno', palette = color_dict, ax = ax[2])
ax[2].plot([0, 1], [0, 1], transform = ax[2].transAxes, ls = '--', c = 'black')

leg = ax[2].legend(title="", loc='lower right', prop={'size': 8})
# Change the legend names. Replace 'new_label_1', 'new_label_2', etc. with your new names.
new_labels = ['Iliac flare ratio']
for t, l in zip(leg.texts[0:], new_labels):  # Skip the title, start from index 1
    t.set_text(l)

r = stats.pearsonr(visit_melt_ratio['value_visit_1'], visit_melt_ratio['value_visit_2'])[0]
ax[2].set_title("Ratio phenotypes\n$r$ = {:.5f}".format(r))
ax[2].set_xlabel('First visit (ratio)')
ax[2].set_ylabel('Second visit (ratio)')
ax[2].set_xlim(min(visit_melt_ratio['value_visit_1'].min(), visit_melt_ratio['value_visit_2'].min()) - 0.01,
                max(visit_melt_ratio['value_visit_1'].max(), visit_melt_ratio['value_visit_2'].max()) + 0.01)                   
ax[2].set_ylim(min(visit_melt_ratio['value_visit_1'].min(), visit_melt_ratio['value_visit_2'].min()) - 0.01,
                max(visit_melt_ratio['value_visit_1'].max(), visit_melt_ratio['value_visit_2'].max()) + 0.01)

plt.tight_layout()

plt.savefig("out_fig/select_two_visit.pdf", bbox_inches='tight')

---

In [ ]:
Image.open("origin_data/816_288_cp/00003.jpg")

In [ ]:
img = Image.open("origin_data/816_288_cp/00003.jpg")
img = img.convert('RGB')
root = "/Users/alexxu/Library/CloudStorage/Box-Box/Narasimhan_lab/hip_shape/"
pred = json.load(open(os.path.join(root, "key_results/all_res_pred_on_pred.json")))
kps = pred['00003_82']
draw = ImageDraw.Draw(img)
for kp in kps:
    draw.ellipse((kp[0]-2, kp[1]-2, kp[0]+2, kp[1]+2), fill = 'red')

In [ ]:
img

In [ ]:
# all prediciton results
all_pred_on_pred = json.load(open('key_results/all_res_pred_on_pred_23.json'))
all_pred_on_anno = json.load(open('key_results/all_res_pred_on_anno_23.json'))

In [ ]:
all_pred_on_pred

In [ ]:
ed = {}
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

for img_id in all_pred_on_pred.keys():
    ed[img_id] = {}
    for idx in range(len(hip_regions)):
        ed[img_id][hip_regions[idx]] = cal_euclidean_distance(all_pred_on_pred[img_id][idx][0], 
                                                              all_pred_on_pred[img_id][idx][1],
                                                              all_pred_on_anno[img_id][idx][0],
                                                              all_pred_on_anno[img_id][idx][1])

In [ ]:
len(ed.keys())

In [ ]:
len(ed['15666_82'].keys())

In [ ]:
error_ed_df = pd.DataFrame.from_dict(ed, orient='index')

In [ ]:
error_ed_df

In [ ]:
error_ed_df_melt = error_ed_df.melt()

In [ ]:
error_ed_df_melt

In [ ]:
sns.boxplot(data=error_ed_df_melt, x='value', y='variable')

In [ ]:
error_ed_df.sort_values(by = 'iliac_crest_left', ascending = False).head(20)

In [ ]:
show_kps('00922_93')

### Use manual annotated images

In [ ]:
anno = json.load(open("/Users/alexxu/Library/CloudStorage/Box-Box/Narasimhan_lab/hip_shape/annotation_images/trainval_both_23.json"))

dtypes = {"Image ID": object}
df_82 = pd.read_csv('./annotation_images/816_288_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
df_93 = pd.read_csv('./annotation_images/960_384_Patient_EID_master_list_v2.csv', sep = '\t', dtype = dtypes)
dict_82 = df_82[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()
dict_93 = df_93[['File', "Image ID"]].set_index("File")['Image ID'].to_dict()

In [ ]:
fns = [i['file_name'] for i in anno['images']]

In [ ]:
anno_img_ids = []
for fn in fns:
    if fn in dict_82.keys():
        anno_img_ids.append(dict_82[fn] +'_82')
    else:
        anno_img_ids.append(dict_93[fn] +'_93')

In [ ]:
# all prediciton results
all_pred_on_pred = json.load(open('key_results/all_res_pred_on_pred_23.json'))
all_pred_on_anno = json.load(open('key_results/all_res_pred_on_anno_23.json'))

In [ ]:
all_pred_on_pred

In [ ]:
all_pred_on_pred_flt = {key: all_pred_on_pred[key] for key in anno_img_ids if key in all_pred_on_pred}
all_pred_on_anno_flt = {key: all_pred_on_anno[key] for key in anno_img_ids if key in all_pred_on_anno}

In [ ]:
all_pred_on_pred_flt.keys() == all_pred_on_anno_flt.keys()

In [ ]:
ed = {}
hip_regions = ['iliac_crest_left', 'iliac_crest_right', 'iliac_spine_left', 'iliac_spine_right', 'iliopubic_eminence_left', 'iliopubic_eminence_right', 
               'inferior_pubic_ramus_left', 'inferior_pubic_ramus_right', 'pubic_arch', 'sciatic_notch_left', 'sciatic_notch_right', 'sacrum', 
               'pubic_tubercle', 'trochanter_left', 'trochanter_right', 'obturator_left', 'obturator_right', 
               'sacrum_left', 'sacrum_right', 'inferior_iliac_spine_left', 'inferior_iliac_spine_right','acetabular_inferior_left', 'acetabular_inferior_right']

for img_id in all_pred_on_pred_flt.keys():
    ed[img_id] = {}
    for idx in range(len(hip_regions)):
        ed[img_id][hip_regions[idx]] = cal_euclidean_distance(all_pred_on_pred_flt[img_id][idx][0], 
                                                              all_pred_on_pred_flt[img_id][idx][1],
                                                              all_pred_on_anno_flt[img_id][idx][0],
                                                              all_pred_on_anno_flt[img_id][idx][1])

In [ ]:
error_ed_df = pd.DataFrame.from_dict(ed, orient='index')

In [ ]:
error_ed_df

In [ ]:
error_ed_df_melt = error_ed_df.melt()

In [ ]:
plt.figure(figsize=(5, 10))
sns.boxplot(data=error_ed_df_melt, x='value', y='variable')
plt.xlabel('Euclidean Distance (pixel)')
plt.ylabel('Hip Landmark')

In [ ]:
error_ed_df.sort_values(by='iliac_crest_right', ascending=False)

In [ ]:
def show_kps(image_id):
    # get the files
    all_ppred = json.load(open('key_results/all_res_pred_on_pred_23.json'))
    all_pred = json.load(open('key_results/all_res_pred_on_anno_23.json'))

    img_path = 'images/all_images_cp'
    
    # get image by image_id
    img = Image.open(os.path.join(img_path, image_id + '.jpg'))
    img = img.convert('RGB')
    draw = ImageDraw.Draw(img)

    # get the keypoints
    kps = all_pred[image_id]
    for kp in kps:
        draw.ellipse((kp[0]-2, kp[1]-2, kp[0]+2, kp[1]+2), fill=(0, 255, 0))
    kps = all_ppred[image_id]
    for kp in kps:
        draw.ellipse((kp[0]-2, kp[1]-2, kp[0]+2, kp[1]+2), fill=(255, 0, 0))
    return img

In [ ]:
show_kps("04134_93")

In [ ]:
img_path = 'images/all_images_cp'

# get image by image_id
img = Image.open(os.path.join(img_path, "04134_82" + '.jpg'))

In [ ]:
img